In [0]:
from pyspark.sql import functions as F, Window

catalog = "neobank"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver")


def bronze(t):
    return spark.table(f"{catalog}.bronze.{t}")


def dedupe(df, key, order_col):
    w = Window.partitionBy(key).orderBy(F.col(order_col).desc_nulls_last())

    return df.withColumns({"_rn": F.row_number().over(w)}).filter("_rn = 1").drop("_rn")


def save_silver(df, t):
    df = df.drop("insert_timestamp").withColumns(
        {"update_timestamp": F.current_timestamp()}
    )
    df.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(f"{catalog}.silver.{t}")
    print(f"{t}: {spark.table(f'{catalog}.silver.{t}').count()} rows")

In [0]:
branches = (
    bronze("branches")
    .filter(F.col("branch_code").isNotNull())
    .withColumn("branch_code", F.trim("branch_code"))
    .withColumn("branch_name", F.trim("branch_name"))
    .drop("created_at")
)

save_silver(dedupe(branches, "branch_code", "insert_timestamp"), "branches")

customers = (
    bronze("customers")
    .filter(F.col("customer_id").isNotNull())
    .withColumn("first_name", F.initcap(F.trim("first_name")))
    .withColumn("last_name", F.initcap(F.trim("last_name")))
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("pan_number", F.upper(F.trim("pan_number")))
    .drop("created_at", "updated_At")
)

save_silver(dedupe(customers, "customer_id", "insert_timestamp"), "customers")

accounts = (
    bronze("accounts")
    .filter(F.col("account_id").isNotNull())
    .withColumn("account_type", F.upper(F.trim("account_type")))
    .drop("created_at", "updated_at")
)

save_silver(dedupe(accounts, "account_id", "insert_timestamp"), "accounts")

In [0]:
txns = (
    bronze("transactions")
    .filter(F.col("txn_id").isNotNull())
    .withColumn("txn_type", F.upper(F.trim("txn_type")))
    .drop("created_at")
)

save_silver(dedupe(txns, "txn_id", "txn_timestamp"), "transactions")

logs = bronze("payment_gateway_logs").filter(F.col("txn_id").isNotNull())

save_silver(dedupe(logs, "txn_id", "insert_timestamp"), "payment_gateway_logs")

cbr = bronze("credit_bureau_reports").filter(F.col("customer_id").isNotNull())

w = Window.partitionBy("customer_id", "bureau_pull_date").orderBy(
    F.col("insert_timestamp").desc()
)
cbr = cbr.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

save_silver(cbr, "credit_bureau_reports")

In [0]:
# 1. duplicates on keys (all should be 0)
for t, k in [
    ("customers", "customer_id"),
    ("accounts", "account_id"),
    ("transactions", "txn_id"),
    ("payment_gateway_logs", "txn_id"),
    ("branches", "branch_code"),
]:
    df = spark.table(f"{catalog}.silver.{t}")
    print(t, "dupes:", df.count() - df.select(k).distinct().count())

# 2. orphans: accounts with a branch or customer that doesn't exist
a = spark.table(f"{catalog}.silver.accounts")
print(
    "accounts w/o customer:",
    a.join(
        spark.table(f"{catalog}.silver.customers"), "customer_id", "left_anti"
    ).count(),
)
print(
    "accounts w/o branch:",
    a.join(
        spark.table(f"{catalog}.silver.branches"), "branch_code", "left_anti"
    ).count(),
)

In [0]:
s = lambda t: spark.table(f"{catalog}.silver.{t}")

print(
    "txns w/o account:",
    s("transactions").join(s("accounts"), "account_id", "left_anti").count(),
)
print(
    "logs w/o txn:",
    s("payment_gateway_logs").join(s("transactions"), "txn_id", "left_anti").count(),
)
print(
    "customers w/o branch:",
    s("customers").join(s("branches"), "branch_code", "left_anti").count(),
)
print(
    "bureau w/o customer:",
    s("credit_bureau_reports").join(s("customers"), "customer_id", "left_anti").count(),
)